In [1]:
import json
import os
import ast
from collections import defaultdict
import pandas as pd

In [2]:
# Get list of all files in the directory
input_dir = './../Generation/Filtered_Output/'
if not os.path.exists(input_dir):
    print(f"Directory not found: {input_dir}")
else:
    files = os.listdir(input_dir)
    jsonl_files = [file for file in files if file.endswith('.jsonl') and 'dataset_nl_prompt_best' not in file]
    print(f"Found {len(jsonl_files)} files")

Found 48 files


In [3]:
def check_compilable(code):
    try:
        ast.parse(code)
        return True
    except:
        # Check for Java class structure heuristic
        if 'public class' in code or ('class ' in code and '{' in code and '}' in code):
            return True
        return False

In [4]:
compilation_data = []

for file in jsonl_files:
    file_path = os.path.join(input_dir, file)
    print(f"Processing {file}...")

    # Parse Model and Temp from filename
    name_part = file.replace('.jsonl', '')
    is_github = name_part.startswith('github-dataset')
    if name_part.startswith('github-dataset_java_nl_prompt_best_'):
        remain = name_part.replace('github-dataset_java_nl_prompt_best_', '')
    elif name_part.startswith('dataset_java_nl_prompt_best_'):
        remain = name_part.replace('dataset_java_nl_prompt_best_', '')
    elif name_part.startswith('dataset_nl_prompt_best_'):
        remain = name_part.replace('dataset_nl_prompt_best_', '')
    else:
        remain = name_part

    if '_' in remain:
        model_name, temp = remain.rsplit('_', 1)
    else:
        model_name = remain
        temp = 'N/A'
    if is_github:
        model_name = 'GitHub_' + model_name

    with open(file_path, 'r', encoding='utf-8') as f:
        # Read line by line to handle potential errors gracefully
        lines = f.readlines()

    file_stats = defaultdict(lambda: {"Total": 0, "Compilable_before": 0, "Compilable_after": 0})

    for line in lines:
        if not line.strip(): continue
        try:
            item = json.loads(line)
            
            # Handle 'generations' format
            if 'generations' in item and isinstance(item['generations'], dict):
                for lang, codes in item['generations'].items():
                    for output_obj in codes:
                        if isinstance(output_obj, dict):
                            file_stats[lang]["Total"] += 1
                            
                            original_code = output_obj.get('code', '')
                            # Apply simple cleanup if needed (though analyze_compilability didn't seem to strip markdown)
                            # But repair_evaluation previously stripped markdown. Let's start with raw check as per analyze_compilability.
                            if check_compilable(original_code):
                                file_stats[lang]["Compilable_before"] += 1
                            
                            if output_obj.get('compilable', False):
                                file_stats[lang]["Compilable_after"] += 1
            
            # Handle 'output' format (legacy)
            elif 'output' in item and isinstance(item['output'], list):
                lang = item.get('language', 'Unknown')
                for output_obj in item['output']:
                    if isinstance(output_obj, dict):
                        file_stats[lang]["Total"] += 1
                        
                        original_code = output_obj.get('code', '')
                        if check_compilable(original_code):
                            file_stats[lang]["Compilable_before"] += 1
                        
                        if output_obj.get('compilable', False):
                            file_stats[lang]["Compilable_after"] += 1

        except json.JSONDecodeError:
            pass

    # Aggregate results for this file
    for lang, counts in file_stats.items():
        compilation_data.append({
            "Model": model_name,
            "Temp": temp,
            "Language": lang,
            "Total": counts["Total"],
            "Compilable_before": counts["Compilable_before"],
            "Compilable_after": counts["Compilable_after"],
            "Compilable_before (%)": (counts["Compilable_before"]/ counts["Total"]) * 100 if counts["Total"] > 0 else 0,
            "Compilable_after (%)": (counts["Compilable_after"]/ counts["Total"]) * 100 if counts["Total"] > 0 else 0,
        })

Processing github-dataset_java_nl_prompt_best_starcoder2_0.0.jsonl...
Processing github-dataset_java_nl_prompt_best_gpt-4o-mini_1.0.jsonl...
Processing github-dataset_java_nl_prompt_best_gpt-4o-mini_0.4.jsonl...
Processing github-dataset_java_nl_prompt_best_gpt-4o-mini_0.6.jsonl...
Processing github-dataset_java_nl_prompt_best_starcoder2_0.2.jsonl...
Processing github-dataset_java_nl_prompt_best_starcoder2_0.6.jsonl...
Processing dataset_java_nl_prompt_best_qwen2.5_0.8.jsonl...
Processing github-dataset_java_nl_prompt_best_gpt-4o-mini_0.2.jsonl...
Processing dataset_java_nl_prompt_best_starcoder2_0.8.jsonl...
Processing github-dataset_java_nl_prompt_best_starcoder2_1.0.jsonl...
Processing github-dataset_java_nl_prompt_best_qwen2.5_0.8.jsonl...
Processing github-dataset_java_nl_prompt_best_gpt-4o-mini_0.0.jsonl...
Processing github-dataset_java_nl_prompt_best_starcoder2_0.4.jsonl...
Processing github-dataset_java_nl_prompt_best_gemini-2.5-flash_0.2.jsonl...
Processing dataset_java_nl_pr

In [5]:
df = pd.DataFrame(compilation_data)
if not df.empty:
    # Sort the DataFrame
    df = df.sort_values(by=["Model", "Temp", "Language"])
    
    output_csv = 'compilation_results_multi.csv'
    df.to_csv(output_csv, index=False)
    print(f"Saved results to {output_csv}")
    print(df.head())
else:
    print("No data collected.")

Saved results to compilation_results_multi.csv
                       Model Temp   Language  Total  Compilable_before  \
357  GitHub_gemini-2.5-flash  0.0  Afrikaans    200                 93   
364  GitHub_gemini-2.5-flash  0.0     Arabic    200                 82   
360  GitHub_gemini-2.5-flash  0.0  Bulgarian    200                 92   
355  GitHub_gemini-2.5-flash  0.0    Chinese    200                 93   
361  GitHub_gemini-2.5-flash  0.0      Dutch    200                 84   

     Compilable_after  Compilable_before (%)  Compilable_after (%)  
357               160                   46.5                  80.0  
364               150                   41.0                  75.0  
360               153                   46.0                  76.5  
355               159                   46.5                  79.5  
361               152                   42.0                  76.0  


In [6]:
import pandas as pd
df = pd.DataFrame(compilation_data)
if not df.empty:
    df = df.sort_values(by=['Model', 'Temp', 'Language'])
    df.to_csv('./Result/compilation_results_Java.csv', index=False)
    print('Saved to ./Result/compilation_results_Java.csv')
    print(df.head())
else:
    print('No data collected.')


Saved to ./Result/compilation_results_Java.csv
                       Model Temp   Language  Total  Compilable_before  \
357  GitHub_gemini-2.5-flash  0.0  Afrikaans    200                 93   
364  GitHub_gemini-2.5-flash  0.0     Arabic    200                 82   
360  GitHub_gemini-2.5-flash  0.0  Bulgarian    200                 92   
355  GitHub_gemini-2.5-flash  0.0    Chinese    200                 93   
361  GitHub_gemini-2.5-flash  0.0      Dutch    200                 84   

     Compilable_after  Compilable_before (%)  Compilable_after (%)  
357               160                   46.5                  80.0  
364               150                   41.0                  75.0  
360               153                   46.0                  76.5  
355               159                   46.5                  79.5  
361               152                   42.0                  76.0  
